# Blender tank smoke — mixed run (Kaggle T4)

**Mета:** валідувати повний PR#3+sesja8 config на tank — один прогін з mixed:
occlusion, hard-negatives (empty landscape), wreck-hn — усе разом.

**Distribution grid (57 viable комбінацій із 60):**
- Близька (≥100px, класифікація): 31.6% — 75-100м alt + hfov 30° zoom
- Середня (30-100px, детекція):    47.4% — 150-200м alt + hfov 92°/112°
- Далека (<30px, edge):            21.1% — 300м alt + широкі hfov

**Ratios на n=28:**
- 4 empty landscape (hard_neg, кожен 7-й: 6, 13, 20, 27) — БЕЗ bbox
- 2 wreck-tank (destroyed_proc, mode=hn) — БЕЗ bbox
- 22 tank з OBB — з них ~11 з occluders (visibility ≥25%), ~11 clean

**Що перевіряємо:**
- Preflight `filter_viable` = 57/60
- Preview grid: tanks від 15 до 454 px length (розкид зон)
- OBB коректний під occlusion (amodal — не «пливе»)
- Empty labels у 4 кадрах (hard_neg + wreck)
- `visibility_fraction` < 1.0 у occluded кадрах (Blender 4.2 Specular fix)

**Stage 3-5 (Qwen edit + composite + polish)** — на RunPod A100, не тут.
Kaggle T4 не тягне Qwen 20B (~40GB VRAM). Цей notebook = Stage 1-2 (Blender + AOV).

**Потрібен Kaggle dataset вхід** (+ Add Input):
```
assets/hdri/{summer,autumn_mud,winter,spring}/*.hdr
assets/textures/ground/{summer,autumn_mud,winter,spring}/*_diff_*.{png,jpg}
assets/models/tank/*.glb
assets/models/destroyed_proc/*.glb   (для wreck-hn кадрів)
```

## 1. Discover Kaggle input dataset (auto-detect assets root)

In [ ]:
from pathlib import Path

INPUT = Path('/kaggle/input')
print('Kaggle inputs:')
for d in sorted(INPUT.iterdir()):
    print(' ', d)

# Auto-detect assets root: рекурсивно шукаємо hdri/ у будь-якому dataset.
hdri_dirs = list(INPUT.rglob('hdri'))
if not hdri_dirs:
    raise SystemExit('assets/hdri/ не знайдено — додай Kaggle dataset з assets через + Add Input')
ASSETS = hdri_dirs[0].parent
print(f'\n[assets] {ASSETS}')
for sub in ('hdri', 'textures', 'models'):
    p = ASSETS / sub
    print(f'  {sub}: {"OK" if p.exists() else "MISSING"} → {p}')

tank_models = list((ASSETS / 'models' / 'tank').glob('*.glb')) if (ASSETS / 'models' / 'tank').exists() else []
print(f'\n[tank models] {len(tank_models)}:')
for m in tank_models:
    print(f'  {m.name}')
assert tank_models, 'потрібно ≥1 .glb моделі у assets/models/tank/'

## 2. Install BlenderProc + deps

In [ ]:
!pip install -q blenderproc==2.8.0 'numpy<2' 'opencv-python-headless<4.11' pyyaml pillow matplotlib
# BlenderProc сам стягне Blender 4.x при першому quickstart (~2 хв).
!blenderproc quickstart 2>&1 | tail -5

## 3. Clone repo (main branch)

In [ ]:
import shutil, subprocess
from pathlib import Path

REPO = Path('/kaggle/working/YOLO-Bluebierd')
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run([
    'git', 'clone', '--depth=1', '--branch=main',
    'https://github.com/ChuprinaDaria/YOLO-Bluebierd.git', str(REPO)
], check=True)

print(f'[repo] {REPO}')
for p in sorted((REPO / 'datasetforge' / 'engine').glob('*.py')):
    print(f'  engine/{p.name}')
for p in sorted((REPO / 'datasetforge' / 'configs').glob('*.yaml')):
    print(f'  configs/{p.name}')

## 4. Show config + preflight (single run — використовуємо v1_tank.yaml as-is)

In [ ]:
import sys, yaml
from pathlib import Path

sys.path.insert(0, str(REPO))
from datasetforge.engine.camera_sampler import build_grid_from_config, filter_viable, estimate_target_px

CFG_PATH = REPO / 'datasetforge' / 'configs' / 'v1_tank.yaml'
cfg = yaml.safe_load(CFG_PATH.read_text())

N_FRAMES = 28  # 4 hn + 2 wreck + 22 tank (11 occluded + 11 clean)
print(f'config: {CFG_PATH}')
print(f'n_frames: {N_FRAMES}')
print(f'\nratios:')
print(f"  hard_negatives: {cfg['hard_negatives']['ratio']} → period {round(1/cfg['hard_negatives']['ratio'])} → кожен 7-й empty")
print(f"  destroyed:      {cfg['destroyed']['ratio']} (wreck-hn ~{round(cfg['destroyed']['ratio']*100)}%)")
print(f"  occlusion:      {cfg['occlusion']['ratio']} (~{round(cfg['occlusion']['ratio']*100)}% non-hn з occluders)")

# Preflight — які камери виживають pixel budget?
grid = build_grid_from_config(cfg['camera'])
img_w = cfg['image_size'][0]
tgt = cfg['class']['target_size_m']
ms = cfg['output']['min_side_px']
viable, rej = filter_viable(grid, img_w, tgt, ms)

zones = {'близька (≥100px)': 0, 'середня (30-100px)': 0, 'далека (<30px)': 0}
for s in viable:
    lo, _ = estimate_target_px(s, img_w, tgt)
    if lo >= 100: zones['близька (≥100px)'] += 1
    elif lo >= 30: zones['середня (30-100px)'] += 1
    else: zones['далека (<30px)'] += 1

print(f'\n[grid] viable {len(viable)}/{len(grid)}')
for k, v in zones.items():
    print(f'  {k}: {v} ({100*v/len(viable):.1f}%)')

## 5. Assets — symlink assets root у repo/datasetforge/assets/

In [ ]:
import os, shutil

dst = REPO / 'datasetforge' / 'assets'
if dst.exists() or dst.is_symlink():
    if dst.is_symlink():
        dst.unlink()
    else:
        shutil.rmtree(dst)
os.symlink(str(ASSETS.resolve()), str(dst))
print(f'[symlink] {dst} -> {ASSETS}')

# Sanity — чи бачить engine tank models?
tank_dir = dst / 'models' / 'tank'
print(f'\n[tank models via symlink] {list(tank_dir.glob("*.glb"))}')

## 6. Render mixed (n=28, occlusion + hn + wreck змішано)

In [ ]:
%cd {REPO}
OUT = Path('/kaggle/working/out_tank_mixed')
!MPLBACKEND=Agg blenderproc run datasetforge/engine/render_runner.py \
    --config datasetforge/configs/v1_tank.yaml \
    --n {N_FRAMES} \
    --out {OUT} \
    --assets-root datasetforge/assets \
    --seed 42 2>&1 | tail -60

## 7. Preview grid — контактка з mixed cadres

In [ ]:
SHEET = Path('/kaggle/working/sheet_mixed.jpg')

!python datasetforge/tools/preview_grid.py \
    --out {OUT} \
    --sheet {SHEET} \
    --cols 4 2>&1 | tail -12

from IPython.display import Image, display
if SHEET.exists():
    print(f'\n=== MIXED contact sheet ({SHEET.stat().st_size/1024:.0f} KB) ===')
    display(Image(str(SHEET)))
else:
    print(f'[warn] {SHEET} не знайдено')

## 8. Per-frame breakdown — розкладка типів (hn / wreck / occluded / clean)

In [ ]:
import json
from pathlib import Path

metas = sorted((OUT / 'metadata').glob('*.json'))
print(f'{'stem':<14s} {'type':<12s} {'alt/ang/hfov':<20s} {'est_px':<12s} {'vis':<6s} {'n_occ':<6s} {'zone':<10s}')
print('-' * 90)

n_hn = n_wreck = n_occ_kept = n_clean = 0
for mp in metas:
    m = json.loads(mp.read_text())
    stem = m['image_id']
    is_hn = m.get('is_hard_negative', False)
    is_wreck = m.get('is_destroyed', False)
    vis = m.get('visibility_fraction', 1.0)
    n_o = m.get('n_occluders', 0)
    est = m.get('est_target_px', [0, 0])

    if is_hn:
        typ = 'HARD_NEG'; n_hn += 1
    elif is_wreck:
        typ = 'WRECK'; n_wreck += 1
    elif n_o > 0 and vis < 1.0:
        typ = 'OCCLUDED'; n_occ_kept += 1
    else:
        typ = 'CLEAN'; n_clean += 1

    lo = est[0]
    zone = 'BLIZ' if lo >= 100 else ('MID' if lo >= 30 else ('FAR' if lo > 0 else '-'))
    cam = f"{m.get('altitude_m', 0):.0f}/{m.get('view_angle_deg', 0):.0f}/{m.get('hfov_deg', 0):.0f}"
    est_str = f"{lo:.0f}x{est[1]:.0f}" if est[0] else '-'
    print(f'{stem:<14s} {typ:<12s} {cam:<20s} {est_str:<12s} {vis:<6.2f} {n_o:<6d} {zone:<10s}')

total = len(metas)
print(f'\n== Summary ({total} frames) ==')
print(f'  HARD_NEG (empty landscape): {n_hn} ({100*n_hn/total:.0f}%) — expected ~14% (кожен 7-й)')
print(f'  WRECK (destroyed, no bbox): {n_wreck} ({100*n_wreck/total:.0f}%) — expected ~10%')
print(f'  OCCLUDED (tank + occluders): {n_occ_kept} ({100*n_occ_kept/total:.0f}%)')
print(f'  CLEAN (tank, no occ):        {n_clean} ({100*n_clean/total:.0f}%)')

## 10. Pack output tarball (для download через Kaggle Commit)

In [ ]:
import tarfile

tarball = Path('/kaggle/working/tank_smoke_mixed_2026-07-02.tar.gz')
with tarfile.open(tarball, 'w:gz') as tar:
    tar.add(str(OUT), arcname='out_tank_mixed')
    if SHEET.exists():
        tar.add(str(SHEET), arcname='sheet_mixed.jpg')

size_mb = tarball.stat().st_size / 1024**2
print(f'[tarball] {tarball} ({size_mb:.1f} MB)')
print('Kaggle → Commit → Output tab → download')